In [ ]:
# CPU allocation: set an inclusive range, or leave both as None to retain scheduler affinity.
import os
CPU_START = None
CPU_STOP = None
_available = sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else list(range(os.cpu_count() or 1))
if CPU_START is not None or CPU_STOP is not None:
    if CPU_START is None or CPU_STOP is None:
        raise ValueError('Set both CPU_START and CPU_STOP, or leave both as None.')
    _requested = [cpu for cpu in _available if int(CPU_START) <= cpu <= int(CPU_STOP)]
    if not _requested:
        raise ValueError(f'Requested CPU range [{CPU_START}, {CPU_STOP}] does not intersect {_available}.')
    os.sched_setaffinity(0, set(_requested))
_active_cpus = sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else _available
for _name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_name] = str(max(1, len(_active_cpus)))
os.environ['NUMEXPR_NUM_THREADS'] = str(max(1, min(64, len(_active_cpus))))
print(f'Notebook CPU affinity ({len(_active_cpus)} CPUs): {_active_cpus}')

# Purification Charge-Sharpening Alpha Sweep

Load the saved CPU campaign and produce steady-state and trajectory-resolved entropy and charge-variance figures. This notebook never runs circuit dynamics.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'colab_charge_fluctuations').exists():
    ROOT = ROOT.parent
if not (ROOT / 'colab_charge_fluctuations').exists():
    raise FileNotFoundError('Could not locate repository root containing colab_charge_fluctuations/.')

BUNDLE_ROOT = ROOT / 'colab_charge_fluctuations'
SRC_ROOT = BUNDLE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from charge_sharpening_alpha_sweep_loader import load_charge_sharpening_campaign

KNOWN_CAMPAIGN_IDS = {
    'coarse_dwtrunc1': 'N16_alpha-sweep_nsh1_dwtrunc1_init-maxmix_S10_cycles-2Ny',
    'coarse_dwtrunc0': 'N16_alpha-sweep_nsh1_dwtrunc0_init-maxmix_S10_cycles-2Ny',
    'fine21_dwtrunc1': 'N16_alpha-fine21_nsh1_dwtrunc1_init-maxmix_S10_cycles-2Ny',
}
CAMPAIGN_ID = None  # Or set e.g. KNOWN_CAMPAIGN_IDS['fine21_dwtrunc1'].
payload = load_charge_sharpening_campaign(BUNDLE_ROOT, campaign_id=CAMPAIGN_ID, require_complete=True)
CAMPAIGN_ID = payload['campaign_id']
trajectory_df = payload['trajectory_df']
steady_state_df = payload['steady_state_df']
inventory_df = payload['inventory_df']
critical_diagnostics_df = payload['critical_diagnostics_df']

analysis_root = BUNDLE_ROOT / 'analysis_outputs' / 'purification_charge_sharpening_alpha_sweep_cpu' / CAMPAIGN_ID
figure_root = analysis_root / 'figures'
table_root = analysis_root / 'tables'
figure_root.mkdir(parents=True, exist_ok=True)
table_root.mkdir(parents=True, exist_ok=True)
print(f'Campaign: {CAMPAIGN_ID}')
print(f'Analysis output: {analysis_root}')

## Campaign Completeness

In [ ]:
display(inventory_df)
assert inventory_df['complete'].all(), 'At least one configuration is incomplete.'
assert set(inventory_df['Ny']) == {16, 24, 32}, 'Expected Ny=16,24,32.'
assert inventory_df.query("protocol == 'perfect_correction'")['samples_completed'].eq(10).all()
assert inventory_df.query("protocol == 'postselection'")['samples_completed'].eq(1).all()
inventory_df.to_csv(table_root / 'campaign_inventory.csv', index=False)

## Critical-Point Diagnostics

In [ ]:
display(critical_diagnostics_df)
critical_diagnostics_df.to_csv(table_root / 'alpha2_projector_diagnostics.csv', index=False)

## Derived Tables And Figure Style

In [ ]:
dynamics_df = (
    trajectory_df.groupby(['protocol', 'Ny', 'alpha_topological_region', 'cycle'], as_index=False)
    .agg(
        total_entropy_mean=('total_entropy', 'mean'),
        total_entropy_std=('total_entropy', lambda x: x.std(ddof=1) if len(x) > 1 else 0.0),
        total_charge_variance_mean=('total_charge_variance', 'mean'),
        total_charge_variance_std=('total_charge_variance', lambda x: x.std(ddof=1) if len(x) > 1 else 0.0),
    )
)
steady_state_df.to_csv(table_root / 'steady_state_summary.csv', index=False)
dynamics_df.to_csv(table_root / 'dynamics_summary.csv', index=False)
trajectory_df.to_csv(table_root / 'trajectory_observables_tidy.csv', index=False)

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['CMU Sans Serif', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'legend.fontsize': 7,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
NY_VALUES = tuple(sorted(int(value) for value in steady_state_df['Ny'].unique()))
_NY_PALETTE = ('#0072B2', '#D55E00', '#009E73', '#CC79A7', '#56B4E9')
NY_COLORS = {ny: _NY_PALETTE[index % len(_NY_PALETTE)] for index, ny in enumerate(NY_VALUES)}
PROTOCOL_STYLES = {'perfect_correction': '-', 'postselection': '--'}
PROTOCOL_LABELS = {'perfect_correction': 'perfect correction', 'postselection': 'postselection'}
display(steady_state_df)

## Steady-State Entropy And Charge Variance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.7), constrained_layout=True)
for protocol in ('perfect_correction', 'postselection'):
    for ny in NY_VALUES:
        data = steady_state_df.query('protocol == @protocol and Ny == @ny').sort_values('alpha_topological_region')
        alpha = data['alpha_topological_region'].to_numpy()
        label = rf'$N_y={ny}$, {PROTOCOL_LABELS[protocol]}'
        axes[0].plot(alpha, data['total_entropy_mean'], PROTOCOL_STYLES[protocol], color=NY_COLORS[ny], marker='o', ms=2.5, lw=1.1, label=label)
        axes[1].plot(alpha, data['total_charge_variance_mean'], PROTOCOL_STYLES[protocol], color=NY_COLORS[ny], marker='o', ms=2.5, lw=1.1, label=label)
        if protocol == 'perfect_correction':
            axes[0].fill_between(alpha, data['total_entropy_mean'] - data['total_entropy_std'], data['total_entropy_mean'] + data['total_entropy_std'], color=NY_COLORS[ny], alpha=0.16, linewidth=0)
            axes[1].fill_between(alpha, data['total_charge_variance_mean'] - data['total_charge_variance_std'], data['total_charge_variance_mean'] + data['total_charge_variance_std'], color=NY_COLORS[ny], alpha=0.16, linewidth=0)
for ax in axes:
    ax.axvline(2.0, color='0.35', lw=0.8, ls=':', label=r'$\alpha_1=2$')
    ax.set_xlabel(r'$\alpha_1$')
axes[0].set_ylabel(r'$S_{\mathrm{global}}(2N_y)$')
axes[1].set_ylabel(r'$\delta Q^2(2N_y)$')
axes[0].set_title('Global entropy')
axes[1].set_title('Total charge variance')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=4, frameon=False)
fig.savefig(figure_root / 'steady_state_entropy_charge_variance_vs_alpha.png')
fig.savefig(figure_root / 'steady_state_entropy_charge_variance_vs_alpha.pdf')
plt.show()

## Global Entropy Dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.125, 2.65), sharey=True, constrained_layout=True)
for ax, alpha_value in zip(axes, (1.0, 2.0, 3.0)):
    for protocol in ('perfect_correction', 'postselection'):
        for ny in NY_VALUES:
            data = dynamics_df.query('protocol == @protocol and Ny == @ny and alpha_topological_region == @alpha_value').sort_values('cycle')
            cycles = data['cycle'].to_numpy()
            label = rf'$N_y={ny}$, {PROTOCOL_LABELS[protocol]}'
            ax.plot(cycles, data['total_entropy_mean'], PROTOCOL_STYLES[protocol], color=NY_COLORS[ny], lw=1.1, label=label)
            if protocol == 'perfect_correction':
                ax.fill_between(cycles, data['total_entropy_mean'] - data['total_entropy_std'], data['total_entropy_mean'] + data['total_entropy_std'], color=NY_COLORS[ny], alpha=0.16, linewidth=0)
    ax.set_title(rf'$\alpha_1={alpha_value:g}$')
    ax.set_xlabel('Completed cycle')
axes[0].set_ylabel(r'$S_{\mathrm{global}}$')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3, frameon=False)
fig.savefig(figure_root / 'global_entropy_vs_cycle_alpha_1_2_3.png')
fig.savefig(figure_root / 'global_entropy_vs_cycle_alpha_1_2_3.pdf')
plt.show()

## Total Charge-Variance Dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.125, 2.65), sharey=True, constrained_layout=True)
for ax, alpha_value in zip(axes, (1.0, 2.0, 3.0)):
    for protocol in ('perfect_correction', 'postselection'):
        for ny in NY_VALUES:
            data = dynamics_df.query('protocol == @protocol and Ny == @ny and alpha_topological_region == @alpha_value').sort_values('cycle')
            cycles = data['cycle'].to_numpy()
            label = rf'$N_y={ny}$, {PROTOCOL_LABELS[protocol]}'
            ax.plot(cycles, data['total_charge_variance_mean'], PROTOCOL_STYLES[protocol], color=NY_COLORS[ny], lw=1.1, label=label)
            if protocol == 'perfect_correction':
                ax.fill_between(cycles, data['total_charge_variance_mean'] - data['total_charge_variance_std'], data['total_charge_variance_mean'] + data['total_charge_variance_std'], color=NY_COLORS[ny], alpha=0.16, linewidth=0)
    ax.set_title(rf'$\alpha_1={alpha_value:g}$')
    ax.set_xlabel('Completed cycle')
axes[0].set_ylabel(r'$\delta Q^2$')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3, frameon=False)
fig.savefig(figure_root / 'total_charge_variance_vs_cycle_alpha_1_2_3.png')
fig.savefig(figure_root / 'total_charge_variance_vs_cycle_alpha_1_2_3.pdf')
plt.show()

## Final-Cycle Sharpening Fraction

In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.7), constrained_layout=True)
for protocol in ('perfect_correction', 'postselection'):
    for ny in NY_VALUES:
        data = steady_state_df.query('protocol == @protocol and Ny == @ny').sort_values('alpha_topological_region')
        ax.plot(data['alpha_topological_region'], data['sharpening_fraction'], PROTOCOL_STYLES[protocol], color=NY_COLORS[ny], marker='o', ms=2.5, lw=1.1, label=rf'$N_y={ny}$, {PROTOCOL_LABELS[protocol]}')
ax.axvline(2.0, color='0.35', lw=0.8, ls=':')
ax.set_xlabel(r'$\alpha_1$')
ax.set_ylabel(r'$\Pr[\delta Q^2 < 10^{-2}]$')
ax.set_ylim(-0.03, 1.03)
ax.legend(frameon=False, ncol=2)
fig.savefig(figure_root / 'final_cycle_sharpening_fraction_vs_alpha.png')
fig.savefig(figure_root / 'final_cycle_sharpening_fraction_vs_alpha.pdf')
plt.show()